In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import google.generativeai as genai
import pickle
import warnings
warnings.filterwarnings('ignore')

c:\Users\cdiaziza\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\cdiaziza\AppData\Local\Temp\ipykernel_17468\800248630.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


**Pipeline del modelo / métricas / exportación del modelo**

In [ ]:
class CreditRiskModel:
    def __init__(self, data_path):
        """Inicializa la clase con la ruta de la data y el modelo de ML."""
        self.data_path = data_path
        # Usamos Random Forest - algoritmo robusto
        self.model = RandomForestClassifier(n_estimators=100, random_state=42)
        self.X_train, self.X_test, self.y_train, self.y_test = None, None, None, None

    def prepare_data(self):
        """Carga la data y hace el split de entrenamiento y prueba."""
        df = pd.read_csv(self.data_path)
        
        # X son las variables predictoras, y es el objetivo a predecir
        X = df.drop('loan_status', axis=1)
        y = df['loan_status']
        
        # Separamos 80% para entrenar y 20% para testear
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        print("Data preparada y dividida exitosamente.")

    def train_model(self):
        """Entrena el modelo con los datos de entrenamiento."""
        self.model.fit(self.X_train, self.y_train)
        print("Modelo Random Forest entrenado.")

    def evaluate_model(self):
        """Evalúa el modelo y retorna las métricas offline."""
        predictions = self.model.predict(self.X_test)
        acc = accuracy_score(self.y_test, predictions)
        report = classification_report(self.y_test, predictions)
        return acc, report

    def save_model(self, export_path):
        """Exporta el modelo entrenado como un artefacto .pkl"""
        with open(export_path, 'wb') as f:
            pickle.dump(self.model, f)
        print(f"Modelo exportado en: {export_path}")

In [8]:
# Instanciamos la clase apuntando a nuestra data limpia
ml_pipeline = CreditRiskModel('C:\\Users\\cdiaziza\\Desktop\\projects\\data\\data_clean.csv')

# Ejecutamos los pasos
ml_pipeline.prepare_data()
ml_pipeline.train_model()

# Sacamos las métricas
accuracy, clas_report = ml_pipeline.evaluate_model()
print(f"\nAccuracy del modelo: {accuracy * 100:.2f}%\n")
print("Reporte de Clasificación:\n", clas_report)

# Guardamos el modelo en la carpeta requerida
ml_pipeline.save_model('../artifacts/random_forest_model.pkl')

Data preparada y dividida exitosamente.
Modelo Random Forest entrenado.

Accuracy del modelo: 92.99%

Reporte de Clasificación:
               precision    recall  f1-score   support

         0.0       0.92      0.99      0.96      4963
         1.0       0.96      0.71      0.81      1373

    accuracy                           0.93      6336
   macro avg       0.94      0.85      0.89      6336
weighted avg       0.93      0.93      0.93      6336

Modelo exportado en: ../artifacts/random_forest_model.pkl


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
import google.generativeai as genai

# find_dotenv() busca el archivo .env automáticamente en carpeta de grado superior
_ = load_dotenv(find_dotenv())

# Extraemos la llave
API_KEY = os.getenv("GEMINI_API_KEY")

# Validación estricta: si no hay llave, cortamos la ejecución para que no se estrelle la API
if API_KEY is None:
    raise ValueError(" Aún no lee la llave. Revisa que el archivo se llame estrictamente '.env' (sin .txt oculto)")
else:
    print("Llave cargada correctamente de forma segura.")

def generate_business_insight(acc, report, api_key):
    """Usa Gemini para interpretar los resultados del modelo de ML."""
    genai.configure(api_key=api_key)
    # Usamos el modelo flash del Free Tier
    model = genai.GenerativeModel('gemini-2.5-flash')
    
    prompt = f"""
    Actúa como un analista de datos senior. He entrenado un modelo de Machine Learning 
    para predecir el riesgo crediticio (default vs no default). 
    El modelo obtuvo un Accuracy de {acc:.2f}.
    Aquí está el reporte detallado: {report}
    
    Escribe una conclusión breve (máximo 3 párrafos) explicando estos resultados 
    para un gerente de banco que no sabe de programación.
    """
    
    response = model.generate_content(prompt)
    return response.text

# se genera el insight 
insight = generate_business_insight(accuracy, clas_report, API_KEY)

print("\n--- CONCLUSIÓN GENERADA POR IA ---")
print(insight)

Llave cargada correctamente de forma segura.

--- CONCLUSIÓN GENERADA POR IA ---
Estimado Gerente,

Hemos finalizado el entrenamiento de un modelo de Machine Learning diseñado para predecir el riesgo crediticio, es decir, identificar si un cliente es propenso a caer en *default* o no. En general, el modelo es bastante robusto y **acierta en sus predicciones el 93% de las veces**. Es particularmente fuerte al identificar clientes que *no* representarán un riesgo: de los clientes que efectivamente no incumplieron, el modelo los identificó correctamente en un 99% de las ocasiones. Esto es excelente para agilizar aprobaciones de crédito de bajo riesgo con alta confianza.

Sin embargo, al analizar más a fondo los casos de riesgo, observamos un aspecto importante. Cuando el modelo predice que un cliente *sí* va a caer en *default*, tiene una precisión muy alta del 96%, lo que significa que rara vez nos da una "falsa alarma" sobre un cliente de alto riesgo. La principal área de oportunidad re